#  Gold Layer - Fact Trip

The Fact Trip table is the central table of the star schema.

Unlike dimension tables, which contain descriptive information, the fact table stores the measurable business events.

Each row represents a single taxi trip and references the surrounding dimensions using foreign keys.

### Source

Silver Layer (`taxi.silver.yellow_taxi`)

### Target

`taxi.gold.fact_trip`

### Measures

- Fare Amount
- Tip Amount
- Total Amount
- Trip Distance
- Trip Duration
- Trip Speed
- Passenger Count

### Foreign Keys

- Date Key
- Vendor Key
- Payment Key
- Pickup Location Key
- Dropoff Location Key

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.table("taxi.silver.yellow_taxi")

In [0]:
fact_trip = (
    silver_df

    # Date Key
    .withColumn(
        "date_key",
        F.date_format("pickup_date","yyyyMMdd").cast("int")
    )

    # Vendor Key
    .withColumnRenamed(
        "VendorID",
        "vendor_key"
    )

    # Payment Key
    .withColumnRenamed(
        "payment_type",
        "payment_key"
    )

    # Pickup Location Key
    .withColumnRenamed(
        "PULocationID",
        "pickup_location_key"
    )

    # Dropoff Location Key
    .withColumnRenamed(
        "DOLocationID",
        "dropoff_location_key"
    )
)

In [0]:
window = Window.orderBy(
    "tpep_pickup_datetime",
    "pickup_location_key"
)

fact_trip = fact_trip.withColumn(
    "trip_key",
    F.row_number().over(window)
)

In [0]:
fact_trip = fact_trip.select(

    "trip_key",

    "date_key",

    "vendor_key",

    "payment_key",

    "pickup_location_key",

    "dropoff_location_key",

    "passenger_count",

    "trip_distance",

    "trip_duration_minutes",

    "trip_speed_mph",

    "fare_amount",

    "tip_amount",

    "tip_percentage",

    "total_amount"

)

In [0]:
display(fact_trip)

In [0]:
(
    fact_trip.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("taxi.gold.fact_trip")
)

In [0]:
display(
    spark.table("taxi.gold.fact_trip")
)

In [0]:
print(
    spark.table("taxi.gold.fact_trip").count()
)

In [0]:
%sql
DESCRIBE DETAIL taxi.gold.fact_trip;